# TP 3 : Automatiser le pipeline avec n8n (CORRECTION)

**BUT INFO3 Développeur IA, R5.C.06, DATA5, Semaine 45**
M. Sakkari, IUT de La Rochelle

> **Version enseignant :** réponses aux questions.

## Dans ce TP, on apprend à...
- lancer n8n avec Docker ;
- construire un workflow en reliant des blocs ;
- appeler une API depuis n8n ;
- choisir un chemin selon un résultat (blocs IF) ;
- planifier un workflow ;
- tester les cas de réussite et d'échec.

> **Comment travailler :**
> - Les cellules de code de ce notebook vérifient votre travail. Il n'y a rien à compléter dedans.
> - Le travail se fait dans l'interface de n8n, dans le navigateur.
> - Les tâches « À vous » décrivent ce que le workflow doit faire.

## Le cas
Dans le TP 2, Prefect lançait le pipeline avec du code Python.
Ici, on construit la même automatisation **sans écrire de code** : en reliant des blocs dans n8n.

Le workflow à construire :
1. toutes les minutes, il demande si le fichier `commandes_increment.csv` a changé ;
2. si oui, il lance le pipeline ;
3. il note si le pipeline a réussi ou échoué.

## Comment n8n parle au pipeline
n8n ne sait pas lancer du Python. Il sait appeler une adresse web (une API).
Le fichier `n8n/api_etl.py` est un petit serveur web, déjà écrit. Il expose trois adresses :

| Adresse | Rôle |
|---|---|
| `GET /sante` | répond « ok » si le serveur marche |
| `GET /changement?fichier=...` | dit si le fichier a changé depuis le dernier lancement réussi |
| `POST /lancer` | contrôle le fichier, puis lance le pipeline |

n8n tourne dans Docker. Pour joindre le serveur de votre machine, il utilise l'adresse `host.docker.internal`.

## Partie A : Démarrer les deux serveurs

### À vous : A.1 lancer le serveur du pipeline
1. Ouvrez un terminal dans le dossier du TP (le dossier `TP3_n8n`, obtenu en décompressant le zip `TP3_n8n_etudiants.zip`).
2. Tapez : `python n8n/api_etl.py`.
3. Laissez le terminal ouvert.

### À vous : A.2 lancer n8n
1. Ouvrez Docker Desktop et attendez qu'il soit démarré.
2. Ouvrez un second terminal. Tapez cette commande sur une seule ligne :

`docker run -it --rm --name n8n -p 5678:5678 -e N8N_SECURE_COOKIE=false -v n8n_data:/home/node/.n8n n8nio/n8n`

3. Attendez le message « Editor is now accessible via http://localhost:5678 ».
4. Ouvrez http://localhost:5678 dans votre navigateur.
5. Créez un compte (n'importe quelle adresse et mot de passe : le compte reste sur votre machine).

Que fait chaque morceau de la commande ?

| Morceau | Rôle |
|---|---|
| `-p 5678:5678` | rend n8n joignable sur le port 5678 de votre machine |
| `-v n8n_data:/home/node/.n8n` | garde vos workflows quand le conteneur s'arrête |
| `--rm` | supprime le conteneur à l'arrêt |
| `-e N8N_SECURE_COOKIE=false` | autorise la connexion en http, sans certificat |

In [ ]:
# Cette cellule vérifie que le serveur du pipeline répond. Il n'y a rien à compléter.
import json
import os
import sqlite3
import urllib.request

import outils

reponse = urllib.request.urlopen("http://localhost:8000/sante").read().decode()
print("Le serveur du pipeline répond :", reponse)

**Lecture :** la réponse est `{"statut": "ok"}`. Si la cellule affiche une erreur, le terminal de l'étape A.1 n'est pas lancé.

## Partie B : Un premier workflow
Voir le cours : « Vocabulaire (2/6) », flux de travail, et « Vocabulaire (3/6) », déclencheur.

### À vous : B.1 importer le workflow de départ
1. Dans n8n, cliquez sur « Create workflow ».
2. Cliquez sur le menu `...` en haut à droite, puis « Import from file ».
3. Choisissez `n8n/workflow_depart.json`.

Le workflow contient deux blocs, déjà reliés :
- **Déclencheur** : se déclenche toutes les minutes ;
- **Le fichier a-t-il changé ?** : appelle `GET /changement?fichier=commandes_increment.csv`.

### À vous : B.2 exécuter le bloc
1. Double-cliquez sur le bloc **Le fichier a-t-il changé ?**.
2. Cliquez sur « Execute step ».
3. Lisez la réponse dans la colonne de droite (« Output »).

**Question :** quelle est la valeur du champ `change` ? Pourquoi ?

*Votre réponse :* `true`. Aucun lancement réussi n'a encore été enregistré pour ce fichier : son empreinte est inconnue, donc il est considéré comme changé.

**Question :** dans l'adresse du bloc, pourquoi écrit-on `host.docker.internal` et pas `localhost` ?

*Votre réponse :* n8n tourne dans un conteneur. Pour lui, `localhost` est le conteneur lui-même. `host.docker.internal` désigne la machine qui héberge Docker, où tourne le serveur du pipeline.

## Partie C : Choisir un chemin
### À vous : C.1 lancer le pipeline seulement si le fichier a changé
1. Ajoutez un bloc **If** après « Le fichier a-t-il changé ? ». Nommez-le `Changement ?`.
2. Dans ce bloc, la condition est : le champ `change` (de type Boolean) est `true`.
   - Pour remplir la valeur, faites glisser le champ `change` depuis la colonne de gauche.
3. Ajoutez un bloc **HTTP Request** sur la sortie `true` du bloc If. Nommez-le `Lancer le pipeline`.
   - Method : `POST`.
   - URL : `http://host.docker.internal:8000/lancer`.
   - Cochez « Send Body », Body Content Type : `JSON`, Specify Body : `Using JSON`.
   - Le corps est : `{"fichier": "commandes_increment.csv"}`.
4. Cliquez sur « Execute workflow ».

**Résultat attendu :** le bloc `Lancer le pipeline` renvoie `statut` = `ok`, avec les nombres `lues`, `chargees` et `rejetees`.

In [ ]:
# Cette cellule lit le journal etl_log de la base. Il n'y a rien à compléter.
con = sqlite3.connect("cafe.db")
for ligne in con.execute("SELECT execution, declencheur, lus, charges FROM etl_log ORDER BY rowid DESC LIMIT 3"):
    print("Exécution", ligne[0], "lancée par", ligne[1], ":", ligne[2], "lignes lues,", ligne[3], "chargées.")
con.close()

**Lecture :** la dernière ligne a pour déclencheur `n8n`. C'est votre workflow qui a lancé le pipeline.

**Question :** exécutez le workflow une seconde fois. Que se passe-t-il, et pourquoi ?

*Votre réponse :* le chemin s'arrête au bloc `Changement ?` (sortie `false`). Après le premier lancement réussi, le serveur a enregistré l'empreinte du fichier. Le fichier n'a pas changé, donc le pipeline n'est pas relancé.

## Partie D : Réussite ou échec
Voir le cours : « Vocabulaire (5/6) », relance.

### À vous : D.1 noter le résultat
1. Ajoutez un bloc **If** après `Lancer le pipeline`. Nommez-le `Pipeline réussi ?`.
   - Condition : le champ `statut` (de type String) est égal à `ok`.
2. Ajoutez un bloc **Edit Fields (Set)** sur la sortie `true`. Nommez-le `Succès`.
   - Ajoutez un champ `message` (String) : `Pipeline lancé : les nouvelles commandes sont chargées.`
3. Ajoutez un second bloc **Edit Fields (Set)** sur la sortie `false`. Nommez-le `Échec`.
   - Ajoutez un champ `message` (String) : `Le pipeline a échoué : le fichier source est refusé.`

### À vous : D.2 tester les deux cas
Cas de réussite :
1. Exécutez la cellule ci-dessous : elle ajoute une commande valide au fichier.
2. Exécutez le workflow dans n8n. Le chemin doit finir au bloc `Succès`.

In [ ]:
# Cette cellule ajoute une commande valide à commandes_increment.csv. Il n'y a rien à compléter.
nouvelle = outils.ajouter_commande_valide()
print("La commande", nouvelle, "vient d'être ajoutée à commandes_increment.csv.")

Cas d'échec :
1. Exécutez la cellule ci-dessous : elle crée un fichier `commandes_casse.csv` dont les colonnes sont fausses.
2. Dans n8n, changez le nom du fichier dans les deux blocs HTTP Request : `commandes_casse.csv`.
3. Exécutez le workflow. Le chemin doit finir au bloc `Échec`.
4. Remettez `commandes_increment.csv` dans les deux blocs.

In [ ]:
# Cette cellule crée un fichier de commandes abîmé. Il n'y a rien à compléter.
with open("data/commandes_casse.csv", "w", encoding="utf8") as fichier:
    fichier.write("id;client;montant\n1;2;3\n")
print("Le fichier data/commandes_casse.csv est créé, avec trois colonnes seulement.")

**Lecture :** pour `commandes_casse.csv`, `GET /changement` répond `change` = `true` (fichier jamais lancé).
`POST /lancer` contrôle le fichier, voit que des colonnes manquent, et répond `statut` = `echec` avec la liste des problèmes. Le pipeline n'a pas démarré.

**Question :** que contient le champ `problemes` de la réponse ?

*Votre réponse :* la liste des colonnes manquantes ou des erreurs trouvées par `outils.verifier_source` dans le fichier.

## Partie E : Planifier
Voir le cours : « Vocabulaire (4/6) », planification.

### À vous : E.1 activer le workflow
1. Enregistrez le workflow (`Ctrl+S`).
2. Donnez-lui le nom `Surveillance des commandes`.
3. Activez-le avec l'interrupteur « Active » en haut à droite.
4. Dans le menu de gauche, ouvrez « Executions ». Attendez trois minutes.

**Question :** que voyez-vous dans la liste des exécutions ? Quel chemin suivent-elles ?

*Votre réponse :* une nouvelle exécution toutes les minutes. Elles s'arrêtent au bloc `Changement ?` tant que le fichier ne change pas.

### À vous : E.2 déclencher par un changement
1. Exécutez la cellule D.2 « ajouter une commande valide » une nouvelle fois.
2. Attendez la prochaine exécution, dans « Executions ».
3. Ouvrez-la : le chemin doit passer par `Lancer le pipeline` puis `Succès`.

In [ ]:
# Cette cellule compte les commandes de la base. Il n'y a rien à compléter.
con = sqlite3.connect("cafe.db")
print("La table commande contient", con.execute("SELECT COUNT(*) FROM commande").fetchone()[0], "lignes.")
print("Le pipeline a été lancé par n8n", con.execute("SELECT COUNT(*) FROM etl_log WHERE declencheur = 'n8n'").fetchone()[0], "fois.")
con.close()

Pour finir, **désactivez** le workflow (interrupteur « Active »), sinon il continue de tourner toutes les minutes.

## Bilan
Répondez en une ligne chacune.
1. Quel bloc de votre workflow joue le rôle de **déclencheur** ? De **planification** ?
2. Quelle différence voyez-vous entre Prefect (TP 2) et n8n (TP 3) pour construire la même automatisation ?
3. Pourquoi l'état « le fichier a changé » est-il gardé dans le serveur Python, et pas dans n8n ?
4. Dans quel cas choisiriez-vous Prefect ? Dans quel cas n8n ?

*Votre réponse :* 
1. Le déclencheur est le bloc « Déclencheur » ; la planification est son réglage « toutes les minutes ».
2. Prefect : on écrit du code Python, les tâches sont des fonctions. n8n : on relie des blocs dans une interface, sans code.
3. n8n ne conserve pas ses données entre deux exécutions de test. Le serveur Python garde l'empreinte dans un fichier : elle persiste.
4. Prefect : pipeline en Python, avec des relances et un journal détaillé. n8n : relier des services différents sans écrire de code.